# Results for model: nemotron-3-nano:30b

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
from pathlib import Path

# Load data
df = pl.read_parquet("./jane-street-real-time-market-data-forecasting/train.parquet")

# Calculate global TOP_QUANTILE (top 15%) of 'feature_00' relative to 'responder_6' across rolling batches of 'date_id'
# Step 1: Group by date_id and calculate the 85th percentile of responder_6 per group
# Step 2: Calculate the global threshold as the 85th percentile of these group-level 85th percentiles
# Step 3: Identify rows where feature_00 > global_threshold

# Group by date_id and compute 85th percentile of responder_6 per group
grouped = df.groupby("date_id").agg([
    pl.percentile("responder_6", 0.85).alias("group_85th_responder_6")
])

# Calculate global threshold: 85th percentile of group_85th_responder_6
global_threshold = grouped["group_85th_responder_6"].quantile(0.85)

# Filter rows where feature_00 > global_threshold
top_quantile_mask = df["feature_00"] > global_threshold
top_quantile_df = df.filter(top_quantile_mask)

# Feature Engineering: Create target variable (responder_6) and features (feature_00)
# Note: We use the filtered top_quantile_df for modeling
X = top_quantile_df["feature_00"].to_numpy().reshape(-1, 1)
y = top_quantile_df["responder_6"].to_numpy()

# Train XGBoost Regressor
model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model.fit(X, y)

# Save model for inference (optional, but required for challenge submission)
import joblib
joblib.dump(model, "xgb_model.pkl")